# 01 — Ollama Setup & Core SDK

This notebook covers everything needed to start local LLM inference with Ollama:

1. **Connectivity check** — verify Ollama is running before any call
2. **Model management** — list, inspect, and pull models programmatically
3. **Basic inference** — `ollama.chat()` with conversation history
4. **Model parameters** — temperature, context length, system prompt
5. **OpenAI-compatible endpoint** — drop-in for any OpenAI client (`http://localhost:11434/v1`)
6. **Streaming** — token-by-token output with live throughput metrics

> **Privacy guarantee**: no data leaves this machine. Every call hits `http://localhost:11434`.

In [1]:
from utils.helpers import check_model_available, check_ollama_running

# ── Model used throughout this notebook ────────────────────────────────────
MODEL = "mistral:7b"

# ── Connectivity guard — must pass before any other cell ────────────────────
if not check_ollama_running():
    raise RuntimeError(
        "Ollama is not running.\n"
        "Start it with:  ollama serve\n"
        "Then re-run this cell."
    )

if not check_model_available(MODEL):
    raise RuntimeError(
        f"Model '{MODEL}' is not available locally.\n"
        f"Pull it with:  ollama pull {MODEL}"
    )

print("✓ Ollama is running")
print(f"✓ Model '{MODEL}' is available")

✓ Ollama is running
✓ Model 'mistral:7b' is available


## 1. Model management

`ollama.list()` returns all models currently on disk. Each model object exposes its name,
size on disk, and quantization details.

In [2]:
import ollama

response = ollama.list()

print(f"{'Model':<35} {'Size':>9}  {'Quantization':<16} {'Family'}")
print("─" * 75)
for m in response.models:
    size_gb = m.size / 1e9
    quant = m.details.quantization_level if m.details else "—"
    family = m.details.family if m.details else "—"
    print(f"{m.model:<35} {size_gb:>7.1f} GB  {quant:<16} {family}")

Model                                    Size  Quantization     Family
───────────────────────────────────────────────────────────────────────────
qwen3-embedding:0.6b                    0.6 GB  Q8_0             qwen3
mistral:7b                              4.4 GB  Q4_K_M           llama
qwen3.5:9b                              6.6 GB  Q4_K_M           qwen35
gemma4:e4b                              9.6 GB  Q4_K_M           gemma4
qwen3.5:27b                            17.4 GB  Q4_K_M           qwen35


In [3]:
# Detailed metadata for a single model
info = ollama.show(MODEL)

print(f"Model:          {MODEL}")
print(f"Format:         {info.details.format}")
print(f"Family:         {info.details.family}")
print(f"Parameters:     {info.details.parameter_size}")
print(f"Quantization:   {info.details.quantization_level}")

Model:          mistral:7b
Format:         gguf
Family:         llama
Parameters:     7.2B
Quantization:   Q4_K_M


In [4]:
# Pull a model — safe to call even if already downloaded (no-op after first pull).
# Set DRY_RUN = False to actually download the model.
DRY_RUN = True

if not DRY_RUN:
    print(f"Pulling {MODEL}…")
    ollama.pull(MODEL)
    print("Done.")
else:
    print("Dry run — to pull, set DRY_RUN = False")
    print(f"Equivalent CLI: ollama pull {MODEL}")

Dry run — to pull, set DRY_RUN = False
Equivalent CLI: ollama pull mistral:7b


## 2. Basic inference

`ollama.chat()` is the primary entry point. It takes a model name and a list of messages
in the same format as the OpenAI chat API.

The response object exposes both dict-style (`response['message']['content']`) and
attribute-style (`response.message.content`) access.

In [5]:
from ollama import ChatResponse, chat

response: ChatResponse = chat(
    model=MODEL,
    messages=[
        {"role": "user", "content": "In one sentence, what is Ollama?"}
    ],
)

print(response.message.content)
print("\n── Metadata ──────────────────────────────────")
print(f"Prompt tokens:    {response.prompt_eval_count}")
print(f"Generated tokens: {response.eval_count}")
print(f"Total duration:   {response.total_duration / 1e9:.2f}s")
speed = response.eval_count / (response.eval_duration / 1e9)
print(f"Speed:            {speed:.1f} tokens/sec")

 Ollama is a large language model developed by Mistral AI, based in Paris, that is designed to understand and generate human-like text in multiple languages.

── Metadata ──────────────────────────────────
Prompt tokens:    14
Generated tokens: 34
Total duration:   1.96s
Speed:            50.9 tokens/sec


In [6]:
# Multi-turn conversation — the model has no persistent memory;
# history must be passed explicitly on each call.
messages = [
    {"role": "system", "content": "You are a helpful assistant. Be brief."},
    {"role": "user", "content": "My name is Alice."},
]

r1 = chat(model=MODEL, messages=messages)
print(f"Turn 1 → {r1.message.content.strip()}")

messages.append({"role": "assistant", "content": r1.message.content})
messages.append({"role": "user", "content": "What is my name?"})

r2 = chat(model=MODEL, messages=messages)
print(f"Turn 2 → {r2.message.content.strip()}")

Turn 1 → Hello Alice! How can I assist you today? Let's make this day productive together.
Turn 2 → Your name is Alice.


## 3. Model parameters

Ollama accepts inference parameters via the `options` dict. The most useful ones:

| Parameter | Description | Typical range |
| --- | --- | --- |
| `temperature` | Randomness — lower = more deterministic | 0.0 – 1.0 |
| `num_ctx` | Context window (tokens) | 2048 – 131072 |
| `num_predict` | Max output tokens (-1 = unlimited) | 128 – 4096 |
| `top_p` | Nucleus sampling threshold | 0.5 – 1.0 |
| `repeat_penalty` | Penalise repeated tokens | 1.0 – 1.5 |

In [7]:
response = chat(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are concise. Reply in bullet points only, max 5 bullets.",
        },
        {
            "role": "user",
            "content": "What are the main advantages of running AI models locally?",
        },
    ],
    options={
        "temperature": 0.2,    # low = consistent, focused output
        "num_ctx": 4096,        # context window in tokens
        "num_predict": 300,     # cap output length
    },
)

print(response.message.content)

 - **Performance Efficiency**: Local processing allows for faster computations as there's no need to send data over the internet, reducing latency and improving overall speed.
- **Data Privacy**: By keeping AI models on local devices, sensitive data can be kept secure and private, avoiding potential breaches during data transmission.
- **Offline Capabilities**: Local AI models can function without an internet connection, making them useful in areas with limited or no network access.
- **Reduced Dependency on Cloud Services**: Running AI models locally reduces reliance on cloud providers for computational resources, potentially lowering costs and improving control over the system.
- **Customization and Flexibility**: Local AI models can be tailored to specific use cases and devices, providing a more personalized user experience and enabling quicker updates and adaptations.


## 4. OpenAI-compatible endpoint

Ollama exposes a REST endpoint at `http://localhost:11434/v1` that is fully compatible with
the OpenAI client. This means any library or tool that supports OpenAI can point to Ollama
with a single line change — no API key, no cloud, no cost per token.

In [8]:
from openai import OpenAI

# Point the OpenAI client at the local Ollama server.
# api_key is required by the client library but ignored by Ollama.
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

completion = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Name the three primary colours."}],
    temperature=0.0,
)

print(completion.choices[0].message.content)
print(f"\nTokens — prompt: {completion.usage.prompt_tokens}, "
      f"completion: {completion.usage.completion_tokens}, "
      f"total: {completion.usage.total_tokens}")

 The three primary colors in the RGB (Red, Green, Blue) color model used in digital media are:

1. Red
2. Green
3. Blue

These three primary colors can be combined in various ways to produce a wide range of secondary and tertiary colors. In traditional painting, the primary colors are often considered to be red, yellow, and blue, based on the subtractive color model (cyan, magenta, and yellow).

Tokens — prompt: 10, completion: 100, total: 110


## 5. Streaming

Streaming lets the UI display tokens as they arrive, reducing perceived latency.
Set `stream=True` and iterate over the response generator.

The **final chunk** (`chunk.done == True`) carries the official timing and token counts
straight from Ollama — more accurate than wall-clock measurements.

In [9]:
import time

prompt = "Explain how a transformer neural network works in 5 bullet points."

print(f"Streaming from {MODEL}:\n")
t_start = time.time()
final_chunk = None

stream = chat(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    stream=True,
)

for chunk in stream:
    token = chunk.message.content
    if token:
        print(token, end="", flush=True)
    if chunk.done:
        final_chunk = chunk

elapsed_wall = time.time() - t_start

print("\n\n── Stream stats ──────────────────────────────")
if final_chunk:
    gen_tokens = final_chunk.eval_count
    gen_sec = final_chunk.eval_duration / 1e9
    print(f"Generated tokens: {gen_tokens}")
    print(f"Speed (Ollama):   {gen_tokens / gen_sec:.1f} tokens/sec")
print(f"Wall time:        {elapsed_wall:.2f}s")
print("Data residency:   ✅ Stayed on this machine")

Streaming from mistral:7b:

 1. **Architecture**: A Transformer is a type of deep learning model introduced by Vaswani et al. (2017) for machine translation tasks. It consists of an encoder and decoder, each made up of multiple stacked layers called "Transformer blocks". Each block contains self-attention mechanisms and feed-forward neural networks.

2. **Self-Attention Mechanisms**: The primary innovation in the Transformer model is the self-attention mechanism (also known as scaled dot-product attention). This mechanism allows the model to weigh the importance of words within a sequence when encoding or decoding, enabling it to better capture long-range dependencies compared to traditional Recurrent Neural Networks (RNNs).

3. **Positional Encoding**: To handle positional information since the Transformer is based on pure self-attention and doesn't have any recurrence or convolution, positional encodings are added to the input embeddings. This helps the model capture the order of wor